# UFC fight predictor

predicting who wins a UFC fight using only stuff known before the fight. data is the Ultimate UFC Dataset from kaggle.

went through 5 versions, v5 is the final one. no betting odds as features (only used at the end to compare).

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("archive/ufc-master.csv")
df["date"] = pd.to_datetime(df["date"])

fights = df[df["Winner"].isin(["Red", "Blue"])].copy()
fights["red_win"] = (fights["Winner"] == "Red").astype(int)
fights.shape

(7169, 119)

baseline = predict red's win rate (~58%) for every fight. red is usually the favorite so this is harder to beat than it sounds

## the bug

v2 was beating the baseline until 2020 and then losing every year after. turns out `age_dif` in the dataset flips sign around mid 2020 (R - B before, B - R after). `loss_dif` and `lose_streak_dif` do the same thing.

share of fights where `age_dif` actually equals `B_age - R_age`:

In [2]:
b_minus_r = fights["B_age"] - fights["R_age"]
matches = (fights["age_dif"] == b_minus_r)[b_minus_r != 0]
matches.groupby(fights["date"].dt.year).mean().round(2)

date
2010    0.00
2011    0.00
2012    0.00
2013    0.00
2014    0.00
2015    0.00
2016    0.00
2017    0.00
2018    0.00
2019    0.00
2020    0.75
2021    1.00
2022    1.00
2023    1.00
2024    1.00
2025    1.00
2026    1.00
dtype: float64

so the model learned age backwards for everything after 2020.

v3 = v2 with age recalculated from the raw columns. from v4 on I build all the features myself from the R_/B_ columns instead of trusting their _dif ones

## features

v4/v5 are red minus blue, so positive = red has more.

- win_pct is smoothed, (wins+1)/(fights+2), so a 1-0 guy isn't 100%
- past_prime = years over 32
- debut fighters have no averages, left blank and filled with the median later
- dropped height, reach basically covers it

In [3]:
def fighter_stats(raw, corner):
    p = corner + "_"
    fights_so_far = raw[p + "wins"] + raw[p + "losses"] + raw[p + "draw"]
    debut = fights_so_far == 0
    has_fights = fights_so_far.where(fights_so_far > 0)   # avoid dividing by 0 for debuts
    return pd.DataFrame({
        "log_fights": np.log1p(fights_so_far),
        "debut": debut.astype(int),
        "win_pct": (raw[p + "wins"] + 1) / (fights_so_far + 2),
        "ko_rate": raw[p + "win_by_KO/TKO"] / has_fights,
        "sub_rate": raw[p + "win_by_Submission"] / has_fights,
        "win_streak": raw[p + "current_win_streak"],
        "lose_streak": raw[p + "current_lose_streak"],
        "age": raw[p + "age"],
        "past_prime": (raw[p + "age"] - 32).clip(lower=0),
        "reach": raw[p + "Reach_cms"].where(raw[p + "Reach_cms"] > 0),
        "sig_str": raw[p + "avg_SIG_STR_landed"].where(~debut),
        "sig_str_pct": raw[p + "avg_SIG_STR_pct"].where(~debut),
        "td": raw[p + "avg_TD_landed"].where(~debut),
        "td_pct": raw[p + "avg_TD_pct"].where(~debut),
        "sub_att": raw[p + "avg_SUB_ATT"].where(~debut),
    })


def build_features(raw):
    X = fighter_stats(raw, "R") - fighter_stats(raw, "B")
    X.columns = [c + "_diff" for c in X.columns]
    X["rank_red"] = (raw["better_rank"] == "Red").astype(int)
    X["rank_blue"] = (raw["better_rank"] == "Blue").astype(int)
    return X


model_df = pd.concat([fights[["date", "red_win"]], build_features(fights)], axis=1)

# v1-v3 use the dataset's own difference columns (Blue minus Red)
for col in ["age_dif", "loss_dif", "ko_dif", "sub_dif", "height_dif", "reach_dif", "win_streak_dif"]:
    model_df[col] = fights[col]
model_df["total_fights_dif"] = (fights["B_wins"] + fights["B_losses"]) - (fights["R_wins"] + fights["R_losses"])
model_df["age_dif_fixed"] = fights["B_age"] - fights["R_age"]

v2_features = ["total_fights_dif", "height_dif", "reach_dif", "win_streak_dif", "rank_red", "rank_blue"]
v4_features = ["log_fights_diff", "win_pct_diff", "ko_rate_diff", "sub_rate_diff", "win_streak_diff",
               "lose_streak_diff", "age_diff", "past_prime_diff", "reach_diff", "rank_red", "rank_blue"]
versions = {
    "v1": ["age_dif", "loss_dif", "ko_dif", "sub_dif"],
    "v2": v2_features + ["age_dif"],
    "v3": v2_features + ["age_dif_fixed"],
    "v4": v4_features,
    "v5": v4_features + ["sig_str_diff", "sig_str_pct_diff", "td_diff", "td_pct_diff", "sub_att_diff", "debut_diff"],
}
features = versions["v5"]

train = model_df[model_df["date"] <= "2023-06-03"]
test = model_df[model_df["date"] > "2023-06-03"]
print({name: len(cols) for name, cols in versions.items()})
print(train.shape, test.shape)

{'v1': 4, 'v2': 7, 'v3': 7, 'v4': 11, 'v5': 17}
(5739, 28) (1430, 28)


imputer + scaler + logistic regression in a pipeline so all of it only learns from train

In [4]:
def make_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=1000))

## walk-forward

train on everything before year N, test on year N. only uses fights before the june 2023 cutoff so the test set stays untouched

In [5]:
def walk_forward(feature_cols, data):
    data = data[data["date"] <= "2023-06-03"]          # never touch the test set
    model_scores = []
    base_scores = []
    for i in range(2016, 2024):
        past = data[data["date"].dt.year < i]
        valid = data[data["date"].dt.year == i]
        model = make_model().fit(past[feature_cols], past["red_win"])
        probas = model.predict_proba(valid[feature_cols])[:, 1]
        baseline = [past["red_win"].mean()] * len(valid)
        model_scores.append(log_loss(valid["red_win"], probas))
        base_scores.append(log_loss(valid["red_win"], baseline))
    return pd.Series(model_scores, index=range(2016, 2024)), pd.Series(base_scores, index=range(2016, 2024))


comparison = {}
for name, cols in versions.items():
    comparison[name], base_scores = walk_forward(cols, model_df)
comparison = pd.DataFrame({"baseline": base_scores, **comparison})
comparison.loc["average"] = comparison.mean()
comparison.loc["years beating baseline"] = comparison.drop("average").lt(base_scores, axis=0).sum()
comparison.round(3)

,baseline,v1,v2,v3,v4,v5
2016,0.676,0.653,0.654,0.654,0.653,0.656
2017,0.690,0.661,0.662,0.662,0.659,0.656
2018,0.695,0.671,0.671,0.671,0.665,0.663
2019,0.683,0.676,0.681,0.681,0.683,0.677
2020,0.667,0.714,0.676,0.651,0.650,0.646
2021,0.687,0.739,0.696,0.659,0.660,0.649
2022,0.675,0.704,0.690,0.654,0.646,0.633
2023,0.699,0.719,0.708,0.664,0.662,0.651
average,0.684,0.692,0.680,0.662,0.660,0.654
years beating baseline,0.000,4.000,4.000,8.000,8.000,8.000


v1/v2 fall apart after 2020 because of the bug. v3 fixes it and beats the baseline every year, v5 is the best.

tried C = 0.01, 0.1, 1, barely any difference so left it on default

## test set

fights after june 2023 (1430 fights)

In [6]:
Y_train, Y_test = train["red_win"], test["red_win"]
base_rate = Y_train.mean()

rows = {"baseline": {"log loss": log_loss(Y_test, [base_rate] * len(Y_test)),
                     "accuracy": Y_test.mean()}}
for name, cols in versions.items():
    p = make_model().fit(train[cols], Y_train).predict_proba(test[cols])[:, 1]
    rows[name] = {"log loss": log_loss(Y_test, p), "accuracy": ((p > 0.5) == Y_test).mean()}
pd.DataFrame(rows).T.round(4)

,log loss,accuracy
baseline,0.6864,0.5615
v1,0.7042,0.5594
v2,0.6783,0.5699
v3,0.6474,0.6217
v4,0.6412,0.6287
v5,0.6287,0.6455


v5 from here on

In [7]:
X_train, X_test = train[features], test[features]
model = make_model().fit(X_train, Y_train)
probs = model.predict_proba(X_test)[:, 1]
train_probs = model.predict_proba(X_train)[:, 1]

print("test log loss: ", round(log_loss(Y_test, probs), 4))
print("train log loss:", round(log_loss(Y_train, train_probs), 4))

test log loss:  0.6287
train log loss: 0.6515


### calibration

In [8]:
buckets = pd.cut(probs, [0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1])
pd.DataFrame({"predicted": probs, "red_won": Y_test.to_numpy()}).groupby(buckets, observed=True).agg(
    fights=("red_won", "size"), avg_predicted=("predicted", "mean"), actual=("red_won", "mean")).round(3)

,fights,avg_predicted,actual
"(0.0, 0.3]",40,0.258,0.175
"(0.3, 0.4]",116,0.356,0.328
"(0.4, 0.5]",246,0.455,0.390
"(0.5, 0.6]",378,0.553,0.526
"(0.6, 0.7]",377,0.647,0.663
"(0.7, 0.8]",216,0.745,0.759
"(0.8, 1.0]",57,0.837,0.860


### vs betting odds

converting american odds to probabilities and removing the vig. some recent fights have no odds so only comparing on the ones that do

In [9]:
def implied(odds):
    with np.errstate(divide="ignore"):
        return np.where(odds < 0, -odds / (-odds + 100), 100 / (odds + 100))

red_imp = implied(fights.loc[test.index, "R_odds"].to_numpy())
blue_imp = implied(fights.loc[test.index, "B_odds"].to_numpy())
market = red_imp / (red_imp + blue_imp)
has_odds = ~np.isnan(market)

print("test fights with odds:", has_odds.sum())
print("log loss  v5:", round(log_loss(Y_test[has_odds], probs[has_odds]), 4),
      " market:", round(log_loss(Y_test[has_odds], market[has_odds]), 4),
      " baseline:", round(log_loss(Y_test[has_odds], [base_rate] * has_odds.sum()), 4))
print("accuracy  v5:", round(((probs[has_odds] > 0.5) == Y_test[has_odds]).mean(), 4),
      " market:", round(((market[has_odds] > 0.5) == Y_test[has_odds]).mean(), 4))

test fights with odds: 1207
log loss  v5: 0.6214  market: 0.5814  baseline: 0.6867
accuracy  v5: 0.6479  market: 0.6993


weights (scaled so they're comparable):

In [10]:
weights = pd.Series(model[-1].coef_[0], index=features)
weights.sort_values(key=abs, ascending=False).round(3)

age_diff           -0.255
td_diff             0.191
win_pct_diff        0.108
win_streak_diff     0.099
reach_diff          0.092
rank_red            0.087
past_prime_diff    -0.076
sig_str_pct_diff    0.072
debut_diff         -0.062
rank_blue          -0.059
sig_str_diff        0.059
sub_rate_diff       0.027
td_pct_diff        -0.025
ko_rate_diff        0.025
lose_streak_diff   -0.018
log_fights_diff     0.016
sub_att_diff       -0.010
dtype: float64

## next event

upcoming.csv is the next card after the data ends (april 2026). retraining on all fights for this

In [11]:
final_model = make_model().fit(model_df[features], model_df["red_win"])

upcoming = pd.read_csv("archive/upcoming.csv")
p_red = final_model.predict_proba(build_features(upcoming)[features])[:, 1]
up_red = implied(upcoming["R_odds"].to_numpy())
up_blue = implied(upcoming["B_odds"].to_numpy())

pd.DataFrame({
    "red": upcoming["R_fighter"],
    "blue": upcoming["B_fighter"],
    "v5 red %": (p_red * 100).round(0),
    "v5 blue %": ((1 - p_red) * 100).round(0),
    "market red %": (up_red / (up_red + up_blue) * 100).round(0),
})

,red,blue,v5 red %,v5 blue %,market red %
0,Renato Moicano,Chris Duncan,36.0,64.0,38.0
1,Virna Jandiroba,Tabatha Ricci,47.0,53.0,52.0
2,Abdul Rakhman Yakhyaev,Brendson Ribeiro,96.0,4.0,90.0
3,Ethyn Ewing,Rafael Estevam,39.0,61.0,57.0
4,Tommy McMillen,Manolo Zecchini,77.0,23.0,89.0
5,Jose Delano,Robert Ruchala,65.0,35.0,74.0
6,Guilherme Pat,Thomas Petersen,57.0,43.0,52.0
7,Alessandro Costa,Stewart Nicoll,61.0,39.0,76.0
8,Lando Vannata,Darrius Flowers,52.0,48.0,66.0
9,Alice Pereira,Hailey Cowan,69.0,31.0,53.0


## odds + stats

kept the odds out on purpose so the model doesn't just copy the market. but if you give it the odds, do the stats add anything?

odds only vs odds + v5 features (odds turned into log-odds)

In [12]:
red_imp_all = implied(fights["R_odds"].to_numpy())
blue_imp_all = implied(fights["B_odds"].to_numpy())
market_all = pd.Series(red_imp_all / (red_imp_all + blue_imp_all), index=fights.index)
odds_df = model_df.assign(market_logit=np.log(market_all / (1 - market_all)))
odds_df = odds_df[np.isfinite(odds_df["market_logit"])]

o_train = odds_df[odds_df["date"] <= "2023-06-03"]
o_test = odds_df[odds_df["date"] > "2023-06-03"]
odds_plus = ["market_logit"] + features


def make_odds_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


odds_model = make_odds_model().fit(o_train[["market_logit"]], o_train["red_win"])
both_model = make_model().fit(o_train[odds_plus], o_train["red_win"])

y_odds = o_test["red_win"]
candidates = {
    "market as-is": 1 / (1 + np.exp(-o_test["market_logit"])),
    "odds only": odds_model.predict_proba(o_test[["market_logit"]])[:, 1],
    "odds + stats": both_model.predict_proba(o_test[odds_plus])[:, 1],
    "v5 (stats only)": model.predict_proba(o_test[features])[:, 1],
}
pd.DataFrame({
    "log loss": {k: log_loss(y_odds, p) for k, p in candidates.items()},
    "accuracy": {k: ((p > 0.5) == y_odds).mean() for k, p in candidates.items()},
}).round(4)

,log loss,accuracy
market as-is,0.5814,0.6993
odds only,0.5793,0.7051
odds + stats,0.5747,0.7084
v5 (stats only),0.6214,0.6479


weights:

In [13]:
pd.Series(both_model[-1].coef_[0], index=odds_plus).sort_values(key=abs, ascending=False).round(3).head(10)

market_logit       0.769
td_diff            0.135
age_diff          -0.110
win_pct_diff      -0.102
win_streak_diff    0.066
reach_diff         0.058
sub_att_diff       0.052
rank_blue         -0.042
td_pct_diff       -0.036
sub_rate_diff     -0.032
dtype: float64

is that real or just noise? walk-forward first

In [14]:
rows = []
for i in range(2016, 2024):
    past = o_train[o_train["date"].dt.year < i]
    valid = o_train[o_train["date"].dt.year == i]
    p_odds = make_odds_model().fit(past[["market_logit"]], past["red_win"]).predict_proba(valid[["market_logit"]])[:, 1]
    p_both = make_model().fit(past[odds_plus], past["red_win"]).predict_proba(valid[odds_plus])[:, 1]
    rows.append({"year": i, "odds only": log_loss(valid["red_win"], p_odds),
                 "odds + stats": log_loss(valid["red_win"], p_both)})

odds_wf = pd.DataFrame(rows).set_index("year")
odds_wf.loc["average"] = odds_wf.mean()
print("years where stats helped:", (odds_wf.drop("average")["odds + stats"] < odds_wf.drop("average")["odds only"]).sum(), "of 8")
odds_wf.round(4)

years where stats helped: 6 of 8


,odds only,odds + stats
year,,
2016,0.6225,0.6239
2017,0.6125,0.6025
2018,0.6003,0.5999
2019,0.6319,0.6350
2020,0.6234,0.6176
2021,0.6283,0.6175
2022,0.5977,0.5938
2023,0.5939,0.5847
average,0.6138,0.6094


bootstrap on the test set

In [15]:
def per_fight_loss(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).to_numpy()


gain = per_fight_loss(y_odds, candidates["odds only"]) - per_fight_loss(y_odds, candidates["odds + stats"])
rng = np.random.default_rng(0)
resampled = [gain[rng.integers(0, len(gain), len(gain))].mean() for _ in range(2000)]
print("improvement in log loss:", round(gain.mean(), 4))
print("95% range:", np.percentile(resampled, [2.5, 97.5]).round(4))

improvement in log loss: 0.0045
95% range: [-0.0016  0.0111]


## results

test log loss / accuracy:
- baseline: 0.686 / 56.2%
- v1: 0.704 / 55.9%
- v2: 0.678 / 57.0%
- v3: 0.647 / 62.2%
- v4: 0.641 / 62.9%
- v5: 0.629 / 64.6%
- betting odds: 0.581 / 69.9% (only the 1207 fights with odds, v5 gets 0.621 on those)

fixing the age column did more than any model change. market is still way ahead, it knows stuff this dataset doesn't (injuries, pre-UFC records etc).

odds + stats was slightly better than odds only in 6/8 years but the bootstrap range includes 0, so can't really say the stats add anything to the market.

next: try gradient boosting, find pre-UFC records somewhere